# 인컴 회지로 챗봇 만들기 — 파인튜닝 vs RAG

**INCOM 2학기 첫 수요모임 (2026-09-23)**  
코랩 무료 T4 하나로 ① 회지 전처리 → ② QA 데이터 → ③ Unsloth LoRA 학습 → ④ RAG → ⑤ 둘을 나란히 비교합니다.

> 런타임 유형을 **T4 GPU**로 바꾸고 시작하세요. (런타임 → 런타임 유형 변경 → T4 GPU)  
> 런타임을 재시작했다면 **런타임 → 이전 셀 모두 실행**으로 위에서부터 다시 돌리면 됩니다.

## 0. 준비

In [ ]:
#@title 0-1. 설정
MODEL_NAME = "unsloth/Qwen3.5-4B"            # 대안: "unsloth/Qwen3-4B"
EMBED_NAME = "Qwen/Qwen3-Embedding-0.6B"
REPO_RAW   = "https://raw.githubusercontent.com/2026-INCOM/incom-hoeji-chatbot/main"
ADAPTER_ID = "uuuhyun/incom-hoeji-lora"       # 미리 학습해 둔 LoRA 어댑터 (HF Hub)
MAX_SEQ    = 2048

In [ ]:
#@title 0-2. 설치 (3~6분)
%%capture
import os, importlib.util
!pip install -qqq --upgrade uv
if "3.5" in MODEL_NAME or "3.6" in MODEL_NAME:
    # Qwen3.5 계열은 하이브리드 구조라 전용 커널이 필요 → Unsloth 공식 노트북 설치 블록 그대로
    if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
        try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
        except: _numpy = "numpy"; _pil = "pillow"
        !uv pip install -qqq \
            "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
            "unsloth[base] @ git+https://github.com/unslothai/unsloth"
        !uv pip install -qqq --no-deps "torchcodec==0.7.0"
    elif importlib.util.find_spec("unsloth") is None:
        !uv pip install -qqq unsloth
    !uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
    !uv pip install transformers==5.2.0
    !uv pip uninstall -qqq flash-linear-attention fla-core
    !uv pip install --no-build-isolation causal_conv1d==1.6.0
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"
else:
    !pip install -qqq unsloth
!pip install -qqq faiss-cpu python-docx sentence-transformers matplotlib

In [ ]:
#@title 0-3. GPU 확인
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
#@title 0-4. 데이터 받기 (회칙 + 행사 후기 발췌본)
import os, urllib.request
os.makedirs("data", exist_ok=True)
# 필수: 회지 발췌본. 선택: 미리 만들어 둔 QA(없으면 2-4 셀에서 직접 생성)
for f, required in [("hoeji_excerpt.docx", True), ("qa_train.jsonl", False)]:
    if os.path.exists(f"data/{f}"):
        continue
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/data/{f}", f"data/{f}")
        print("받음:", f)
    except Exception as e:
        if required:
            print(f"{f} 다운로드 실패({e}). 직접 업로드하세요.")
            from google.colab import files
            up = files.upload()
            for name, blob in up.items(): open(f"data/{name}", "wb").write(blob)
        else:
            print(f"{f} 없음 → 2-4 셀에서 생성합니다.")
!ls -la data

## 1. 전처리 — 회지 docx를 "챗봇이 읽을 수 있는 조각"으로

원본 회지는 구글독스 10MB(이미지 27장 + 회원 명부)입니다. 오늘은 **회칙 + 행사 후기**만 잘라낸 38KB 발췌본을 씁니다.  
전처리의 목표는 하나: **"질문 하나에 답이 들어 있는 크기"로 문서를 쪼개기.**

In [ ]:
#@title 1-1. docx → 문단 리스트
from docx import Document
doc = Document("data/hoeji_excerpt.docx")
paras = [p.text for p in doc.paragraphs]
print("문단 수:", len(paras))
for p in paras[:12]:
    print(repr(p[:70]))          # repr로 보면 탭·빈 줄·끝 공백이 그대로 보입니다

In [ ]:
#@title 1-2. 청소: 빈 줄 · 탭 · 끝 공백
import re
def clean(s):
    s = s.replace("\u3000", " ").replace("\t", " ")
    s = re.sub(r"[ ]{2,}", " ", s)
    return s.strip()
lines = [clean(p) for p in paras]
lines = [l for l in lines if l]
print(f"{len(paras)} → {len(lines)} 줄")

In [ ]:
#@title 1-3. 섹션 단위로 자르기 (제목 + 작성자 줄이 구분자)
AUTHOR = re.compile(r"^4\d기\s")            # "46기 성소민", "45기 회장 박상준"
CHAPTER = re.compile(r"^제\s?\d+장")          # 회칙: "제 1장 : 총칙"
SKIP = {"연간 행사", "INCOM 회칙"}

sections, cur = [], None
i = 0
while i < len(lines):
    l = lines[i]
    nxt = lines[i+1] if i+1 < len(lines) else ""
    if l in SKIP:
        i += 1; continue
    if len(l) <= 15 and AUTHOR.match(nxt):            # 행사 후기 제목
        cur = {"title": l, "author": nxt, "body": []}; sections.append(cur); i += 2; continue
    if CHAPTER.match(l):                               # 회칙 장
        cur = {"title": "INCOM 회칙 " + l, "author": "", "body": []}; sections.append(cur); i += 1; continue
    if cur is None:
        cur = {"title": "기타", "author": "", "body": []}; sections.append(cur)
    cur["body"].append(l); i += 1

print("섹션 수:", len(sections))
for s in sections: print(f"- {s['title']}  ({s['author']})  {sum(map(len, s['body']))}자")

In [ ]:
#@title 1-4. 너무 긴 섹션은 더 쪼개기 → chunks (앞 조각 끝을 조금 겹침)
import json
MAX_CHARS = 700      # 조각 하나의 최대 길이
OVERLAP   = 150      # 같은 섹션 안에서 쪼갤 때, 앞 조각의 끝 150자를 다음 조각 머리에 붙임

chunks = []
for s in sections:
    head = s["title"] + (f" (작성: {s['author']})" if s["author"] else "")
    buf, prev_tail = [], ""
    for para in s["body"] + [None]:
        if para is None or sum(map(len, buf)) + len(para) > MAX_CHARS:
            if buf:
                body = ("…" + prev_tail + "\n" if prev_tail else "") + "\n".join(buf)
                chunks.append({"id": len(chunks), "title": s["title"], "text": head + "\n" + body})
                prev_tail = "\n".join(buf)[-OVERLAP:].lstrip()     # 다음 조각에 넘길 꼬리
            buf = []
        if para is not None: buf.append(para)

print("청크 수:", len(chunks))
two = [c for c in chunks if c["text"].split("\n")[1].startswith("…")]
print(f"앞 조각과 겹치는 청크: {len(two)}개")
print("\n--- 겹침 예시 ---\n" + two[0]["text"][:420])
json.dump(chunks, open("data/chunks.json", "w"), ensure_ascii=False, indent=1)

## 2. QA 데이터 — 학습시키려면 "질문-답" 쌍이 필요하다

파인튜닝은 문서를 그냥 먹이는 게 아니라 **대화 형태의 예시**를 먹입니다. 청크마다 LLM에게 QA를 만들어 달라고 합니다.

데이터를 만드는 건 큰 모델(코랩 내장 Gemini, 키 필요 없음)에게 시키고, 학습은 작은 모델(Qwen 4B)에게 시킵니다. 실전에서도 흔한 조합입니다.

In [ ]:
#@title 2-1. 모델 로드 (Unsloth, 4bit)
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
)
tok = getattr(tokenizer, "tokenizer", tokenizer)   # 멀티모달 모델이면 processor 안의 tokenizer
FastLanguageModel.for_inference(model)
print("GPU 사용량:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

In [ ]:
#@title 2-2. chat() 헬퍼
SYSTEM = "너는 인하대학교 컴퓨터 동아리 INCOM(인컴)의 안내 챗봇이야. 한국어로, 친근한 존댓말로 답해."

def chat(user, system=SYSTEM, max_new_tokens=300, sample=False):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=sample,
                             temperature=0.7 if sample else None, top_p=0.9 if sample else None,
                             pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>\s*", "", ans, flags=re.S).strip()

print(chat("인컴 수요모임이 뭐야?"))     # 아직 아무것도 모르는 상태

In [ ]:
#@title 2-3. 청크 하나로 QA 몇 쌍 만들어 보기
QA_PROMPT = """너는 인하대 컴퓨터 동아리 INCOM(인컴)의 안내 챗봇 학습 데이터를 만드는 중이야.
아래 [글]만 근거로, 신입 부원이 챗봇에게 물어볼 법한 질문과 챗봇의 답을 {n}개 만들어.

규칙
1. 챗봇은 제3자다. "저는", "나는", "지원하게 되었습니다" 같은 글쓴이 1인칭 문장은 금지.
   - 행사 구성·진행 순서·강연 내용·규칙처럼 누구에게나 같은 사실은 출처 없이 바로 말한다.
   - 글쓴이 개인의 경험·감상·팀 사정(어려웠다, 팀이 어떻게 구성됐다, 참여 계기)만 "{author}의 후기에 따르면 ~라고 해요"로 출처를 붙인다.
2. 글쓴이 한 사람의 경험을 동아리 전체의 규칙처럼 일반화하지 마. (어떤 팀이 전기전자공학부 2명이었다고 해서 "팀은 전기전자공학부 2명으로 구성됩니다"라고 쓰면 안 됨)
3. 답은 글에 있는 사실만, 2~3문장, 친근한 존댓말.
4. JSON 배열만 출력: [{{"q": "...", "a": "..."}}, ...]

예시
{example}

[글] (작성자: {author})
{text}"""

EX_REVIEW = """Q: 인컴톤은 어떤 순서로 진행돼요?
A: 인컴톤은 강연(기획/개발) → 아이디어톤 → 개발 기간 → 발표 순서로 진행돼요. 강연을 먼저 다 듣고 아이디어를 구체화하는 흐름이에요.
Q: 개발 경험이 없어도 인컴톤에 나갈 수 있어요?
A: 네, 46기 서준영 부원의 후기에 따르면 그 팀은 웹이나 앱을 제대로 만들어 본 사람이 없었는데도 참여했다고 해요. 강연을 들으면서 시작해도 괜찮아요."""
EX_RULE = """Q: 인컴 정기 모임은 언제 해요?
A: 인컴 회칙 제3조에 따르면 매주 수요일에 정기 모임(수요 모임)을 해요. 회칙에 정해진 공식 활동이에요."""

BAD = re.compile(r"(저는|나는|내가|제가|우리 팀|우리 조|지원하게 되었|참여하게 되었|느꼈습니다|기억에 남습니다|배웠습니다|좋았습니다)")

def chunk_author(chunk):
    head = chunk["text"].splitlines()[0]          # 예: "인컴톤 (작성: 46기 서준영)"
    if "(작성: " in head:
        return head.split("(작성: ")[1].rstrip(")") + " 부원"
    return "INCOM 회칙"

QA_ENGINE = "gemini"   # "gemini": 코랩 내장 Gemini (키 불필요, 월 무료 한도) / "local": 위에서 로드한 Qwen

def llm_for_qa(prompt):
    if QA_ENGINE == "gemini":
        from google.colab import ai
        try:
            return ai.generate_text(prompt, model_name="google/gemini-2.5-flash")
        except Exception as e:
            print("gemini-2.5-flash 실패 → flash-lite로 재시도:", e)
            return ai.generate_text(prompt, model_name="google/gemini-2.5-flash-lite")
    return chat(prompt, system=None, max_new_tokens=900)

def make_qa(chunk, n=5):
    author = chunk_author(chunk)
    example = EX_RULE if author == "INCOM 회칙" else EX_REVIEW
    raw = llm_for_qa(QA_PROMPT.format(n=n, author=author, example=example, text=chunk["text"]))
    try:
        items = json.loads(raw[raw.find("["): raw.rfind("]") + 1])   # JSON 배열 부분만
    except Exception:
        print("파싱 실패:", raw[:300]); return []
    out = []
    for x in items:
        if not (x.get("q") and x.get("a")): continue
        if BAD.search(x["a"]): continue                      # 1인칭 잔재는 버림
        out.append({"q": x["q"].strip(), "a": x["a"].strip(), "chunk": chunk["id"]})
    return out

demo = next(c for c in chunks if "인컴톤" in c["title"])
for qa in make_qa(demo, 5):
    print("Q:", qa["q"]); print("A:", qa["a"]); print()

In [ ]:
#@title 2-4. 전체 QA (약 200쌍) — 이미 만들어 둔 파일 로드, 없으면 생성 (Gemini 약 5분 / 로컬 Qwen 약 30분)
QA_PATH = "data/qa_train.jsonl"
if os.path.exists(QA_PATH):
    qa_data = [json.loads(l) for l in open(QA_PATH)]
else:
    qa_data = []
    for c in chunks:
        qa_data += make_qa(c, 5)          # 5개 생성 → 1인칭 필터 후 3~5개, 53청크면 약 200쌍
        print(f"{c['id']+1}/{len(chunks)} 누적 {len(qa_data)}쌍")
    with open(QA_PATH, "w") as f:
        for x in qa_data: f.write(json.dumps(x, ensure_ascii=False) + "\n")
print("QA 쌍:", len(qa_data))
for x in qa_data[:3]: print("Q:", x["q"], "\nA:", x["a"], "\n")

## 3. 파인튜닝 — Unsloth로 LoRA 학습

Unsloth는 LoRA 학습을 **2배 빠르게, 메모리는 절반**으로 돌려주는 라이브러리입니다. 무료 T4에서 4B 모델을 학습할 수 있는 이유가 이겁니다.

In [ ]:
#@title 3-1. LoRA 어댑터 붙이기
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 3407, max_seq_length = MAX_SEQ,
)
model.print_trainable_parameters()

In [ ]:
#@title 3-2. 학습 데이터 포맷 (대화 → 텍스트)
from datasets import Dataset
def to_text(x):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": x["q"]}, {"role": "assistant", "content": x["a"]}]
    try:    return {"text": tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}
    except TypeError: return {"text": tok.apply_chat_template(msgs, tokenize=False)}
train_ds = Dataset.from_list(qa_data).map(to_text)
print(train_ds[0]["text"])

In [ ]:
#@title 3-3. 학습 함수 (숫자 세 개만 봅시다: 배치 · 학습률 · 스텝)
from trl import SFTTrainer, SFTConfig

def reset_lora():
    # 앞 실험에서 발산(nan)한 어댑터를 그대로 이어 쓰지 않도록, 매번 A·B를 초기값으로 되돌림
    for m in model.modules():
        if hasattr(m, "reset_lora_parameters"):
            m.reset_lora_parameters("default", True)

def run_training(batch, lr, max_steps=60):
    global last_trainer
    reset_lora()
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(
        model = model, tokenizer = tok, train_dataset = train_ds,
        args = SFTConfig(
            dataset_text_field = "text", max_seq_length = MAX_SEQ,
            per_device_train_batch_size = batch, gradient_accumulation_steps = 1,
            warmup_steps = 5, max_steps = max_steps, learning_rate = lr,
            logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01,
            lr_scheduler_type = "linear", seed = 3407, output_dir = "outputs", report_to = "none",
        ),
    )
    last_trainer = trainer          # 중간에 멈춰도 3-7에서 곡선을 그릴 수 있게
    trainer.train()
    return trainer

In [ ]:
#@title 3-4. 학습 시작
trainer = run_training(batch = 256, lr = 1e-2)

> 런타임을 재시작했다면: 커서를 **3-4에 두고** 런타임 → 이전 셀 모두 실행 (0-1 ~ 3-3), 그다음 3-5부터 실행.

In [ ]:
#@title 3-5. 배치를 줄여서 다시  (loss가 nan이 되면 ■로 멈추고 다음 셀로)
trainer = run_training(batch = 2, lr = 1e-2, max_steps = 30)

In [ ]:
#@title 3-6. 학습률도 줄여서 다시
trainer = run_training(batch = 2, lr = 2e-4)

In [ ]:
#@title 3-7. loss 곡선
import matplotlib.pyplot as plt
t = last_trainer                  # 직전에 돌린(중간에 끊었어도) 학습
losses = [x["loss"] for x in t.state.log_history if "loss" in x]
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss")
plt.title(f"batch={t.args.per_device_train_batch_size}, lr={t.args.learning_rate}"); plt.show()

In [ ]:
#@title 3-8. 미리 3 epoch 돌려둔 어댑터 불러오기
FastLanguageModel.for_inference(model)
try:
    model.load_adapter(ADAPTER_ID, adapter_name="full")
    model.set_adapter("full")
    print("사전 학습 어댑터 로드 완료:", ADAPTER_ID)
except Exception as e:
    print("어댑터 로드 실패 → 방금 학습한 60스텝 어댑터로 계속합니다.", e)

print(chat("인컴 수요모임이 뭐야?"))

In [ ]:
#@title (사전 실행용) 3-9. 전체 학습 3 epoch + 저장/업로드 — 세션 중에는 건너뜀
FULL_TRAIN = False
if FULL_TRAIN:
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds,
        args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_SEQ,
            per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=10,
            num_train_epochs=3, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit",
            weight_decay=0.01, lr_scheduler_type="linear", seed=3407, output_dir="outputs_full", report_to="none"))
    trainer.train()
    model.save_pretrained("lora_full"); tok.save_pretrained("lora_full")
    # from huggingface_hub import login; login()
    # model.push_to_hub(ADAPTER_ID); tok.push_to_hub(ADAPTER_ID)

## 4. RAG — 외우게 하지 말고 찾아보게 하기

질문이 오면 ① 관련 청크를 검색해서 ② 그 내용을 프롬프트에 붙여 ③ 모델이 "자료를 보고" 답하게 합니다. 모델은 그대로, 지식은 바깥에.

In [ ]:
#@title 4-1. 청크 임베딩 → FAISS 인덱스
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
embedder = SentenceTransformer(EMBED_NAME, device="cuda", model_kwargs={"torch_dtype": torch.float16})
doc_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=8, show_progress_bar=True)
index = faiss.IndexFlatIP(doc_vecs.shape[1]); index.add(doc_vecs.astype("float32"))
print("인덱스 크기:", index.ntotal, "| 벡터 차원:", doc_vecs.shape[1])

In [ ]:
#@title 4-2. 검색해 보기
def retrieve(q, k=3):
    qv = embedder.encode([q], prompt_name="query", normalize_embeddings=True)
    scores, ids = index.search(qv.astype("float32"), k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], ids[0])]

for score, c in retrieve("인컴톤이 뭐야?"):
    print(f"[{score:.3f}] {c['title']}  →  {c['text'][:80]}...")

In [ ]:
#@title 4-3. 검색 결과를 붙여서 답하기
RAG_PROMPT = """아래 [자료]만 근거로 질문에 답해. 자료에 없는 내용이면 "회지에 그 내용은 없어요"라고 말해.

[자료]
{ctx}

[질문] {q}"""

def rag_answer(q, k=3):
    ctx = "\n\n".join(f"({i+1}) {c['text']}" for i, (s, c) in enumerate(retrieve(q, k)))
    with model.disable_adapter():                       # RAG는 원본 모델로
        return chat(RAG_PROMPT.format(ctx=ctx, q=q))

print(rag_answer("인컴톤이 뭐야?"))

## 5. 대결 — 같은 질문, 세 가지 답

- **base**: 아무것도 안 한 원본 모델  
- **FT**: 회지 QA 200쌍으로 LoRA 학습한 모델  
- **RAG**: 원본 모델 + 검색한 회지 청크

In [ ]:
#@title 5-1. 세 방식 나란히
def ask_base(q):
    with model.disable_adapter(): return chat(q)
def ask_ft(q): return chat(q)
def ask_rag(q): return rag_answer(q)

QUESTIONS = [
    "인컴 회칙 제1조에서 정한 동아리의 정식 영어 명칭은?",
    "회칙이 마지막으로 개정된 건 언제고 몇 차 개정이야?",
    "자료편집부가 맡는 일 세 가지가 뭐야?",
    "46기 인컴톤 후기는 누가 썼고 어떤 내용이야?",
    "인컴 동아리방은 어디에 있어?",          # 회지에 없는 질문
]
for q in QUESTIONS:
    print("=" * 80); print("Q:", q)
    for name, fn in [("base", ask_base), ("FT  ", ask_ft), ("RAG ", ask_rag)]:
        print(f"\n[{name}] {fn(q)}")

### 정리

| | 파인튜닝 (LoRA) | RAG |
|---|---|---|
| 배우는 것 | 말투·형식·역할 | (배우지 않음) |
| 사실 정확도 | 세부 사실은 자주 지어냄 | 검색만 맞으면 정확, 근거 제시 가능 |
| 자료가 바뀌면 | 다시 학습 | 파일만 교체 |
| 잘 맞는 일 | 특정 스타일·포맷·도메인 말투 | 사내 문서 QA, 최신 정보 |
| 실전 | **둘을 같이** — RAG로 사실, 가벼운 FT로 말투 | |

In [ ]:
#@title 6. (덤) 채팅 UI
!pip install -qqq gradio
import gradio as gr
gr.ChatInterface(lambda m, h: rag_answer(m), title="인컴 회지 챗봇 (RAG)").launch(share=False, debug=False)